# PerfectPetClub - Data Engineering ETL Pipeline
This notebook  Extract, Transform and Loads supplied database.

### The pipeline:
Extracts records from a CSV file.

Cleans and standardises extracted data.

Exports the cleaned dataset in CSV and Parquet formats.

Displays the original and transformed datasets for validation.


In [ ]:
from pathlib import Path
import sys
import pandas as pd

sys.path.insert(0, str(Path("../src").resolve()))

%load_ext autoreload
%autoreload 2


In [ ]:

from crm import run_crm_etl

# Call the function to run  crm etl file 
crm_df, cleaned_crm_df,crm_features_df = run_crm_etl()

# Display the first five rows to confirm to check extraction and transformation 
display(crm_df.head())

display(cleaned_crm_df.head())

display(crm_features_df.head())

In [ ]:

 
# Test: Can our validation detect an invalid date?

test_data = raw_customers.copy()

# Deliberately insert an invalid subscription date
test_data.loc[0, 'sub_end_date'] = 'invalid-date'

try:
    validate_customer_data(test_data)
    print("TEST FAILED: Invalid date was accepted.")

except AssertionError as e:
    print("TEST PASSED: Invalid date was detected.")
    print(e)



 

2026-11-11 16:51:38,643 - INFO - Executing automated Data Quality checks on customer subscriptions...


TEST PASSED: Invalid date was detected.
DQ FAIL: Missing or invalid subscription dates.


/Users/roshangauchan/Documents/DataEngineering/DataEngineeringPresentation/src/extract_customer_subscriptions.py:101: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  


In [ ]:
# 1. Transform your customer and subscription data
clean_customers = transform_customer_features(raw_customers)

# 2. Remove unnecessary identifiers
final_dataset = clean_customers.drop(
    columns=['customer_name', 'postcode', 'subscription_id']
)

# 3. Check your processed data
print("Total customers:", len(final_dataset))
print("Churned customers:", final_dataset['is_churned'].sum())
print("Dataset shape:", final_dataset.shape)

print(f'Master feature dataset shape: {final_dataset.shape}')
final_dataset.head()


2026-11-11 16:51:51,505 - INFO - Applying feature engineering on customer subscriptions...
2026-11-11 16:51:51,527 - INFO - Target Engineered: 86/221 customers churned (38.91%).


Total customers: 221
Churned customers: 86
Dataset shape: (221, 11)


,customer_id,email,county,sub_start_date,sub_end_date,auto_renew,subscription_tier,delivery_service,is_churned,subscription_tenure_days,postcode_district
0,1,Sargsyan701@somemail.co.uk,Berkshire,2026-01-01 00:00:00.000,2026-01-31 00:00:00.000,0,SuperSaver,Economy72,1,30,SL6
1,2,Mammadova282@example.com,Oxfordshire,2026-01-01 00:00:00.000,2026-11-27 00:00:00.000,1,SuperSaver,Economy72,0,330,OX7
2,3,Sana858@notrealemail.com,Worcestershire,2026-01-01 00:00:00.000,2026-11-27 00:00:00.000,1,Premium,SignedFor24,0,330,DY10
3,4,Omar.Farooq4@example.com,Kent,2026-01-01 00:00:00.000,2026-11-27 00:00:00.000,1,Premium,SignedFor24,0,330,TN9
4,5,Alejandro357@somemail.co.uk,Hertfordshire,2026-01-01 00:00:00.000,2026-05-01 00:00:00.000,0,SuperSaver,Economy72,1,120,WD7


In [ ]:

# Churn rate by subscription tier
tier_summary = final_dataset.groupby('subscription_tier')['is_churned'].agg(
    total_customers='count',
    churn_rate='mean'
).reset_index()

# Convert churn rate into percentage
tier_summary["churn_rate"] = (
    tier_summary["churn_rate"] * 100
).round(2)

# Churned vs Retained spend & review comparison
print('\n--- Mean Metrics by Churn Status ---')
print(final_dataset.groupby('is_churned')[['total_spend_gbp', 'total_orders', 'avg_star_rating', 'one_star_reviews_count']].mean().round(2))
Stage 4: Load — Export Curated Dataset to Apache Parquet
As specified in Task 2, the final transformed dataset must be exported as an Apache Parquet file. Parquet provides columnar compression, preserves column data types, and is optimized for downstream analytical queries.



--- Churn Rate by Subscription Tier ---


,subscription_tier,total_customers,churn_rate
0,Basic,77,33.77
1,Premium,47,40.43
2,SuperSaver,97,42.27


In [ ]:
from pathlib import Path
import pandas as pd

# Create output folder
output_dir = Path("../output")
output_dir.mkdir(parents=True, exist_ok=True)

# Save only my customer and subscription features
parquet_path = output_dir / "customer_subscription_features.parquet"

# Export to Parquet
final_dataset.to_parquet(parquet_path, index=False)

print(f'✓ Successfully loaded {len(final_dataset)} records into Parquet format:')
print(f'  Location: {parquet_path.resolve()}')
print(f'  File size: {parquet_path.stat().st_size:,} bytes')


In [ ]:
# Verify the new customer-only Parquet file
df_verified = pd.read_parquet(parquet_path)

print("Parquet verification passed!")
print("Rows:", df_verified.shape[0])
print("Columns:", df_verified.shape[1])
print("Columns match:", list(df_verified.columns) == list(final_dataset.columns))

In [ ]:
# Final verification of feature engineering and Parquet output

# 2. Check customer count has not changed
assert len(final_dataset) == len(raw_customers)

# 3. Verify churn calculation
expected_churn = (
    final_dataset["auto_renew"] == 0
).astype(int)

assert (final_dataset["is_churned"] == expected_churn).all()

# 4. Check subscription durations
assert (final_dataset["subscription_tenure_days"] >= 0).all()

# 5. Verify exported Parquet structure
assert df_verified.shape == final_dataset.shape
assert list(df_verified.columns) == list(final_dataset.columns)

print("PASS: Customer records are unique")
print("PASS: Customer count is correct")
print("PASS: Churn calculations are correct")
print("PASS: Subscription durations are valid")
print("PASS: Parquet structure is verified")